> ## SOLUTION / ANSWER KEY &mdash; Lab 10.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-debug-and-fix.ipynb`](../lab-10-debug-and-fix.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.10 &mdash; Debug & Fix the Loop

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Diagnose a wrong-tool + ungrounded bug from a captured trace
- Apply the fix: give the agent a read-only grounding tool
- Run the FIXED agent for real and verify it grounds

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it &amp; observe**. The responsible-AI logic here &mdash; injection defence, least privilege, trace-reading, fairness, the eval loop, the guardrails &mdash; is **real, deterministic Python** you can run offline. The **Advanced agent labs (10&ndash;12)** additionally drive a **real Groq model** through `create_agent`: **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a responsible agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The agent labs use a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`); the guardrail/eval/trace logic is genuine rule-based Python. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Debugging, in code](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-10-10")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=5):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The full debug loop (deck slides 14&ndash;16): read the trace to **localise** the bug, **diagnose** the
failure mode, **fix** at the right layer, and **verify** with a re-run. Here you first diagnose a **captured
buggy run** (a real message list, recorded so the diagnosis is deterministic): it called a tool it wasn't
given (**wrong tool**) and then computed on an **ungrounded** number. The fix is to give the agent a
**grounding** tool &mdash; then you run the fixed agent **for real** against Groq and confirm it grounds.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

@tool
def extract_figure(name: str) -> dict:
    """Ground a figure with its source from the filing. Use for any revenue/figure lookup."""
    return {"revenue": {"value": 120.0, "source": "p4"}}.get(name, {})

@tool
def compute(expression: str) -> str:
    """Do exact arithmetic on a numeric expression such as 0.15*120."""
    try:
        return str(safe_calc(expression))
    except Exception:
        return "error: not a numeric expression"

from langchain_core.messages import AIMessage, ToolMessage

# A CAPTURED buggy run (a real message list), recorded once so the diagnosis is deterministic:
BUGGY_TRACE = [
    AIMessage(content="", tool_calls=[{"name": "lookup_order", "args": {"q": "revenue"}, "id": "a"}]),  # wrong tool
    ToolMessage(content="unknown tool: lookup_order", tool_call_id="a"),
    AIMessage(content="", tool_calls=[{"name": "compute", "args": {"expression": "0.15*100"}, "id": "b"}]),  # 100 ungrounded
    ToolMessage(content="15.0", tool_call_id="b"),
    AIMessage(content="~15M (ungrounded)"),
]
GROUNDED = {"120"}   # the only figure actually retrievable from the filing (extract_figure -> value 120)
print("captured buggy trace ready | read-only tools:", extract_figure.name, "&", compute.name)

## Build it
Complete `diagnose` and `ungrounded_compute` (read the captured trace), `final_of`, and give the
`fixed_agent` the **grounding** tool it was missing. The buggy trace is recorded; the fix you'll run live.

> **Bridge from Lab 10.3:** there you read a trace as `(tool, arg, observation)` tuples; here it is the
> **same reads over real objects** &mdash; an `AIMessage` carries `tool_calls` (name + `args`), a
> `ToolMessage` carries the observation.

In [ ]:
from langchain_core.messages import AIMessage
from langchain.agents import create_agent

def tools_used(messages):
    return [tc['name'] for m in messages for tc in (getattr(m, 'tool_calls', None) or [])]

def diagnose(messages):
    # read the trace: what went wrong? (scan each message content for the symptom)
    for m in messages:
        if "unknown tool" in str(getattr(m, "content", "")):
            return "wrong_tool"
    return "ok"

def ungrounded_compute(messages, grounded):
    # a compute tool-call whose expression uses no grounded value (same idea as Lab 10.3)
    for m in messages:
        for tc in (getattr(m, 'tool_calls', None) or []):
            if tc["name"] == "compute" and not any(g in str(tc["args"].get("expression", "")) for g in grounded):
                return True
    return False

def final_of(messages):
    for m in reversed(messages):
        if isinstance(m, AIMessage) and str(getattr(m, 'content', '')).strip():
            return m.content
    return None

# Workflow (create_agent -> CompiledStateGraph) -- the FIXED agent loop:
#
#   START -> model <--> tools -> END
#           (the fix: bind the read-only GROUNDING tool the buggy run was missing)
#
#   Bound: extract_figure (grounding) + compute   (READ-ONLY)
def fixed_agent():
    # the fix: give the agent the read-only GROUNDING tool it was missing, alongside compute
    return create_agent(llm, [extract_figure, compute])

try:
    # Diagnose the CAPTURED buggy run (no model call needed):
    print("buggy tools     :", tools_used(BUGGY_TRACE))
    print("diagnosis       :", diagnose(BUGGY_TRACE))
    print("buggy final     :", final_of(BUGGY_TRACE))
    print("buggy ungrounded?:", ungrounded_compute(BUGGY_TRACE, GROUNDED))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the FIXED agent for real. It should call `extract_figure` first (grounding), then `compute` on 120 &mdash; the bug gone.

_This calls the real `openai/gpt-oss-20b` via Groq. If `GROQ_API_KEY` is unset the cell prints how to set it instead of crashing._

In [ ]:
if not groq_ready():
    print("No GROQ_API_KEY -- add it to the repo .env (free key at console.groq.com), then re-run this cell.")
else:
    try:
        # Now RUN the FIXED agent for real: it should ground via extract_figure, then compute on 120.
        agent = fixed_agent()
        result = with_backoff(lambda: agent.invoke(
            {"messages": [("user", "Use extract_figure to get revenue, then compute 15% of it. Cite the page.")]},
            config={"recursion_limit": 8}))
        print_trace(result)
        print("---")
        print("fixed tools     :", tools_used(result["messages"]))
        print("grounded now?   :", not ungrounded_compute(result["messages"], GROUNDED))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The **captured** buggy trace diagnoses as `wrong_tool` and shows an ungrounded compute (`0.15*100`) &mdash; deterministic, no LLM.
- The **real** fixed agent grounds via `extract_figure` (value 120) *before* computing, so `ungrounded_compute` is now False.
- Read &rarr; diagnose &rarr; fix at the right layer (add the grounding tool) &rarr; verify with a live re-run. That's the debug loop, end to end.

## Your turn (open task &mdash; no grader)
Ask the fixed agent a slightly different grounded question (e.g. *"compute 25% of revenue and cite the
page"*) and re-run. **What good looks like:** the trace still shows `extract_figure` firing before `compute`,
and the compute expression contains the grounded `120`. If it ever computes on an invented number, your
`ungrounded_compute` catches it &mdash; that's the guardrail you just built.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
try:
    if groq_ready():
        agent = fixed_agent()
        result = with_backoff(lambda: agent.invoke(
            {"messages": [("user", "Use extract_figure to get revenue, then compute 25% of it. Cite the page.")]},
            config={"recursion_limit": 8}))
        print_trace(result)
        print("tools used   :", tools_used(result["messages"]))     # extract_figure fires BEFORE compute
        print("grounded now?:", not ungrounded_compute(result["messages"], GROUNDED))
    else:
        print("(add GROQ_API_KEY to .env to run the fixed agent for real)")
except Exception as e:
    print("(Live model hiccup -- a rate limit or a stray built-in tool call. Re-run in a moment.)", type(e).__name__)

---
### Key takeaway
Run -> read the trace -> diagnose -> fix at the right layer -> verify. The buggy run called a tool it lacked and computed on an ungrounded number; giving the real agent a grounding tool fixed both, and the live re-run proved it. That's the debug loop.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>